# 各プレイヤーのボロノイ領域のクラスタリングを行う

## パス関係の定義

In [1]:
import sys
from pathlib import Path

# src ディレクトリのパスを取得して追加
src_dir = str(Path().resolve().parents[1])
if src_dir not in sys.path:
    sys.path.append(src_dir)

## データの取り出し

In [2]:
from analyzers import datamanager

data_manager = datamanager.DataManager("../processed_data")

trajectories = data_manager.get_all_trajectories()

from analyzers import utils, calculator

max_value = calculator.max_recursive(trajectories)
graph_ylim = utils.round_up(max_value)

max_len = calculator.max_len_recursive(trajectories)
graph_xlim = max_len

Match Type:  odict_keys(['AB', 'BD', 'CB', 'G1', 'G2', 'G3', 'G4', 'after', 'before'])


### クラスタリング処理用のメソッド作成（クラスタ数を複数で行うため）

In [3]:
import math
import numpy as np
import matplotlib.pyplot as plt

from analyzers.clustering import drawer, stats

def clustering(
    data: list[np.ndarray],
    n_clusters: int,
    labels: list[str],
    match_type: str,
    player_name: str,
    **kwargs
):
    size_of_ax = (15, 7)
    num_axes = n_clusters
    if num_axes > 2:
        num_cols = 3
    else:
        num_cols = num_axes
    num_rows = math.ceil(num_axes / num_cols)
    fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(size_of_ax[0]*num_cols, size_of_ax[1]*num_rows))

    axes = axes.flatten()

    drawer.plot_kmedoids(
        data=data,
        axes=axes[:n_clusters],
        labels=labels,
        n_clusters=n_clusters,
        **kwargs,
    )

    for j in range(num_axes, len(axes)):
        fig.delaxes(axes[j])

    current_title = fig._suptitle.get_text() if fig._suptitle else ""
    new_title = f"Clustering {player_name} trajectory in {match_type} ({current_title})"
    fig.suptitle(new_title)

    results_dir = Path(f'./results/{n_clusters}_clusters/{match_type}')
    results_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(f'./results/{n_clusters}_clusters/{match_type}/{player_name}.png', bbox_inches='tight')
    # plt.show()
    plt.close()


### クラスタリング処理

クラスタ数は2から9の間で全て処理

In [4]:
all_player_names = data_manager.get_all_player_names()

for player_name in all_player_names:
    player_trajectories = data_manager.get_data_by_player(player_name)
    
    for match_type, games in player_trajectories.items():
        data = list(games.values())
        labels = list(games.keys())

        for i in range(2, 10):
            clustering(
                data=data,
                n_clusters=i,
                labels=labels,
                match_type=match_type,
                player_name=player_name,
                xlim=(0, graph_xlim),
                ylim=(0, graph_ylim),
            )

### チーム毎でクラスタリング処理

In [5]:
all_team_names = data_manager.get_all_teams()
all_team_data = {}

for team_name in all_team_names:
    team_trajectories = data_manager.get_data_by_team(team_name)

    # グラフのリミット決定のための保存
    all_team_data[team_name] = team_trajectories

from analyzers import calculator

y_lim = calculator.max_recursive(all_team_data)

# クラスタリング処理
for team_name, team_trajectories in all_team_data.items():
    for match_type, games in team_trajectories.items():
        data = list(games.values())
        labels = list(games.keys())

        for i in range(2, 10):
            clustering(
                data=data,
                n_clusters=i,
                labels=labels,
                match_type=match_type,
                player_name=team_name,
                xlim=(0, graph_xlim),
                ylim=(0, y_lim),
            )